# Fundamentals 13 - Multi-agent graph portable

Objetivo: encadenar dos agents mediante `toolkit.graph`, conservar ambos `RunResult` y producir una salida auditable sin ejecucion manual alternativa.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_GRAPH_ENGINE | auto | Resolver LangGraph o backend portable. |
| AGENTIC_SYSTEMS_DEMO_SYMBOL | graph | Estado de usuario que inicia el flujo. |
| agents | inspector y reviewer | Ejecutar nodos reales y componer sus RunResult. |

## 1) Topologia

```text
START - inspect - review - END
```

El primer agente observa la API instalada. El segundo revisa esa evidencia. Los nodos no conocen el backend graph.

In [ ]:
import os

import agentic_systems as toolkit

GRAPH_ENGINE = os.getenv("AGENTIC_SYSTEMS_GRAPH_ENGINE", "auto")
SYMBOL_TO_INSPECT = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "environment")
runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

## 2) Tools y agents

Las tools reciben datos del estado. Ninguna contiene una respuesta esperada fija.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.PUBLIC_API}

@toolkit.tool
def review_observation(symbol: str, is_public: bool) -> dict:
    return {
        "symbol": symbol,
        "decision": "accepted" if isinstance(is_public, bool) else "rejected",
        "observed_public_value": is_public,
    }

inspector = system.agent(
    name="multi_graph_inspector",
    instructions="Inspecciona el simbolo solicitado.",
    tools=[inspect_public_api],
    runtime=runtime,
)
reviewer = system.agent(
    name="multi_graph_reviewer",
    instructions="Revisa la observacion estructurada recibida.",
    tools=[review_observation],
    runtime=runtime,
)

## 3) Adaptar agents a nodos

Los callbacks mapean estado de dominio; `toolkit.agent_node` conserva la ejecucion y la normalizacion.

In [ ]:
def inspect_input(state: dict) -> dict:
    return {"tool": "inspect_public_api", "input": {"symbol": state["symbol"]}}


def inspect_output(result, state: dict) -> dict:
    return {
        **state,
        "inspection_run": result,
        "inspection": toolkit.agent_output(result, kind="graph_node"),
    }


def review_input(state: dict) -> dict:
    fields = state["inspection"]["fields"]
    return {
        "tool": "review_observation",
        "input": {"symbol": fields["symbol"], "is_public": fields["is_public"]},
    }


def review_output(result, state: dict) -> dict:
    return {
        **state,
        "review_run": result,
        "review": toolkit.agent_output(result, kind="graph_node"),
    }

inspect_node = toolkit.agent_node(inspector, input=inspect_input, output=inspect_output, result_key=None)
review_node = toolkit.agent_node(reviewer, input=review_input, output=review_output, result_key=None)

## 4) Construir y ejecutar

La fachada es la misma para ambos backends. No hay `if langgraph` ni un `for node` alternativo.

In [ ]:
app = toolkit.graph(
    name="multi_agent_public_api_graph",
    engine=GRAPH_ENGINE,
    state=dict,
    nodes={"inspect": inspect_node, "review": review_node},
    edges=[("START", "inspect"), ("inspect", "review"), ("review", "END")],
)

final_state = app.run({"symbol": SYMBOL_TO_INSPECT})
result = toolkit.compose_result(
    text=f"Inspeccion y revision completadas para {SYMBOL_TO_INSPECT}.",
    data={
        "symbol": SYMBOL_TO_INSPECT,
        "inspection": final_state["inspection"],
        "review": final_state["review"],
    },
    results=[final_state["inspection_run"], final_state["review_run"]],
    mode="graph",
    input={"symbol": SYMBOL_TO_INSPECT},
    meta={"graph_engine": app.engine},
)

toolkit.human_result(result, title="Multi-agent Graph RunResult", show_lineage=True)
toolkit.show_json({
    "resolved_engine": app.engine,
    "inspection": final_state["inspection"],
    "review": final_state["review"],
}, title="Estado final")

## 5) Cobertura real

Ambos resultados internos alimentan `compose_result`; la composicion no fabrica metadata.

In [ ]:
api_coverage = [
    "toolkit.runtime",
    "toolkit.system",
    "toolkit.tool",
    "system.agent",
    "toolkit.agent_node",
    "toolkit.agent_output",
    "toolkit.graph",
    "GraphApp.run",
    "toolkit.compose_result",
    "toolkit.human_result",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Multi-agent graph API coverage")

## Resultado esperado

El estado final conserva inspeccion y revision. El `RunResult` compuesto agrega sus tool events y metadata observada, independientemente del backend graph.